[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/12-generation.ipynb)

# Module 12: Generation

Companion notebook to [Module 12](https://transformers-and-math.drmhse.com/modules/12-generation) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## Part 1 — A small decoder with an optional KV cache (sections 12.1 and 12.2)
One attention layer written so it can run on the whole sequence, or on one new token with stored keys and values.

In [ ]:
class Attn(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dh = h, d // h
        self.qkv, self.o = nn.Linear(d, 3 * d), nn.Linear(d, d)
    def forward(self, x, cache=None):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        if cache is not None:
            if 'k' in cache:
                k, v = torch.cat([cache['k'], k], 2), torch.cat([cache['v'], v], 2)
            cache['k'], cache['v'] = k, v
        causal = cache is None or T > 1
        y = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
        return self.o(y.transpose(1, 2).reshape(B, T, -1))

class Decoder(nn.Module):
    def __init__(self, V=50, d=64, h=4, L=2, ctx=256):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(ctx, d)
        self.attn = nn.ModuleList([Attn(d, h) for _ in range(L)])
        self.ffn = nn.ModuleList([nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)) for _ in range(L)])
        self.ln = nn.ModuleList([nn.LayerNorm(d) for _ in range(2 * L + 1)])
        self.head = nn.Linear(d, V)
    def forward(self, ids, caches=None, offset=0):
        x = self.tok(ids) + self.pos(torch.arange(offset, offset + ids.shape[1]))   # the true positions
        for i, (a, f) in enumerate(zip(self.attn, self.ffn)):
            x = x + a(self.ln[2 * i](x), None if caches is None else caches[i])
            x = x + f(self.ln[2 * i + 1](x))
        return self.head(self.ln[-1](x))

model = Decoder().eval()
ids = torch.randint(0, 50, (1, 24))

### Test the cache the right way: compare logits under teacher forcing

In [ ]:
with torch.no_grad():
    full = model(ids)                                   # all positions in one pass
    caches = [{} for _ in model.attn]
    step_logits = [model(ids[:, :1], caches, 0)]
    for t in range(1, ids.shape[1]):
        step_logits.append(model(ids[:, t:t + 1], caches, offset=t))
    cached = torch.cat(step_logits, 1)
print('largest logit difference, cached vs uncached:', (full - cached).abs().max().item())

## Part 2 — Decoding strategies (section 12.3)

In [ ]:
def top_k(logits, k):
    cut = torch.topk(logits, k).values[-1]
    return logits.masked_fill(logits < cut, float('-inf'))

def top_p(logits, p):
    probs, order = torch.softmax(logits, -1).sort(descending=True)
    keep = probs.cumsum(-1) - probs < p                 # keep tokens until the running total reaches p
    out = torch.full_like(logits, float('-inf'))
    out[order[keep]] = logits[order[keep]]
    return out

def repetition_penalty(logits, seen, theta=1.15):
    l = logits.clone()
    l[seen] = torch.where(l[seen] > 0, l[seen] / theta, l[seen] * theta)
    return l

logits = torch.log(torch.tensor([0.5, 0.3, 0.1, 0.05, 0.05]))
print('top-p 0.9 keeps', torch.isfinite(top_p(logits, 0.9)).sum().item(), 'tokens; renormalized:', torch.softmax(top_p(logits, 0.9), -1).round(decimals=3))
for T in (0.1, 1.0, 3.0):
    print(f'temperature {T}:', torch.softmax(logits / T, -1).round(decimals=3))

## Your turn
1. Remove `offset=t` from the cached loop. How large does the logit difference get, and why (section 12.2)?
2. Time generating 200 tokens with and without the cache for this model. Why is the gap small at this size?